In [ ]:
import os
import subprocess
import tempfile
from datasets import load_from_disk
import llvmlite.binding as llvm
from trl import GRPOTrainer, GRPOConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
import random
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
llvm.initialize()

raw = load_from_disk('processed_data')
dataset_small = raw.select(range(min(8, len(raw))))

def make_prompt(example):
    return (
        """
You are a compiler optimization assistant.
Optimize the following LLVM IR and return only the optimized IR wrapped within <llvm> and </llvm> tags, without any comments or explanations.

Original IR:
{src}

Optimized IR (include <llvm> tags):""".format(src=example['non_optimized']))

dataset = dataset_small.map(
    lambda ex: {'prompt': make_prompt(ex), 'completion': ex['LLVM_instcombine']},
    remove_columns=['LLVM_instcombine', 'prompt'],
)


def reward_func(prompts, completions, completion_ids=None, alive_tv_path='./alive-tv', **kwargs):
    """
    Check for <llvm> tags; if missing, penalty -1. Otherwise use Alive2 + length bonus.
    """
    rewards = []
    for p, c in zip(prompts, completions):
        # enforce <llvm> </llvm> tags
        if '<llvm>' not in c or '</llvm>' not in c:
            rewards.append(-1.0)
            continue
        try:
            src_ir = p.split('Original IR:')[1].split('Optimized IR')[0]
            tgt_ir = c.split('<llvm>')[1].split('</llvm>')[0]
            # Alive2 equivalence
            with tempfile.TemporaryDirectory() as tmp:
                src_file = os.path.join(tmp, 'src.ll')
                tgt_file = os.path.join(tmp, 'tgt.ll')
                open(src_file, 'w').write(src_ir)
                open(tgt_file, 'w').write(tgt_ir)
                proc = subprocess.Popen([alive_tv_path, src_file, tgt_file], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
                out, err = proc.communicate(timeout=10)
                alive_ok = (proc.returncode == 0)
        except Exception:
            alive_ok = False
        if not alive_ok:
            rewards.append(-1.0)
            continue
        # compute length bonus
        try:
            def count_ir(text):
                mod = llvm.parse_assembly(text)
                mod.verify()
                return sum(1 for fn in mod.functions if not fn.is_declaration for bb in fn.blocks for _ in bb.instructions)
            bonus = float(count_ir(src_ir) - count_ir(tgt_ir))
        except Exception:
            bonus = 0.0
        rewards.append(1.0 + bonus + random.randit(10))
    return rewards


config = GRPOConfig(
    output_dir='outputs/qwen2.5-coder-grpo-small',
    use_cpu=True,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    num_generations=2,
    steps_per_generation=1,
    max_prompt_length=4096,
    max_completion_length=16,
    learning_rate=1e-6,
    remove_unused_columns=False,
    num_iterations=1,
    epsilon=0.2,
    beta=0.0,
    scale_rewards=False,
    logging_steps=1,
    max_steps = 10,
    log_completions = True,
    num_completions_to_print=1,
    num_train_epochs=10
)

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-Coder-1.5B-Instruct', padding_side='left')
model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    torch_dtype='auto', use_cache=False,
)
tokenizer.pad_token = tokenizer.eos_token

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()


INFO 07-22 19:27:04 [__init__.py:244] Automatically detected platform cpu.


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: eliray (eliray-queen-mary-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/Users/emilfahretdinov/LLM_for_Compiler/venv/lib/python3.10/site-packages/transformers/trainer.py:3710: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  ctx_manager = torch.cpu.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)
